# WLASL Text-to-Pose — End-to-End Pipeline

ASL үг → 50 frame pose sequence үүсгэх Transformer декодер.

**Шаардлагатай:**
- `data/WLASL_v0.3.json`
- `data/pose_per_individual_videos/` хавтас

**Алхамууд:**
1. Орчин шалгах
2. Дата уншиж судлах
3. Загвар сургах
4. Inference + GIF үүсгэх


## 1. Орчин шалгах

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
from src import config as C

print(f"DATA_DIR:   {C.DATA_DIR}")
print(f"META_FILE:  {C.META_FILE}  exists={C.META_FILE.exists()}")
print(f"POSE_DIR:   {C.POSE_DIR}  exists={C.POSE_DIR.exists()}")
print(f"NUM_WORDS:  {C.NUM_WORDS}")
print(f"SEQ_LEN:    {C.SEQ_LEN}")
print(f"POSE_DIM:   {C.POSE_DIM}  (= {C.NUM_KEYPOINTS} keypoints x {C.KP_DIM})")


## 2. Дата судлах

In [ ]:
from src.dataset import select_top_words

top_words = select_top_words(C.META_FILE, C.NUM_WORDS, C.MIN_VIDEOS_PER_WORD)
for gloss, instances in top_words:
    print(f"{gloss:20s}  {len(instances)} videos")


In [ ]:
from src.dataset import build_index, WLASLPoseDataset

samples, word_to_id, id_to_word = build_index(C.POSE_DIR, top_words)
print(f"Pose folders found: {len(samples)}")
print(f"Vocab: {word_to_id}")


In [ ]:
ds = WLASLPoseDataset(samples)
print(f"Valid sequences: {len(ds)}")

if len(ds) > 0:
    wid, pose = ds[0]
    print(f"Word: {id_to_word[int(wid)]}")
    print(f"Pose tensor shape: {tuple(pose.shape)}")
    print(f"Range: [{pose.min():.3f}, {pose.max():.3f}]")


In [ ]:
import numpy as np
from src.visualize import render_animation, pose_tensor_to_numpy

if len(ds) > 0:
    wid, pose = ds[0]
    pose_np = pose_tensor_to_numpy(pose.numpy())
    out = render_animation(
        pose_np,
        C.OUT_DIR / f"sample_{id_to_word[int(wid)]}.gif",
        title=f"Real: {id_to_word[int(wid)]}",
    )
    print(f"Saved: {out}")


Дээрх GIF-ийг `outputs/` хавтаснаас нээж шалгана уу — хэрэв хүн жинхэнэ дохио хийж байгаа харагдвал дата зөв ачааллагдсан гэсэн үг.

## 3. Сургах

In [ ]:
from src.train import train

model, history, word_to_id, id_to_word = train(verbose=True)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["train"], label="train")
if history["val"]:
    ax.plot(history["val"], label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(C.OUT_DIR / "loss_curve.png", dpi=100)
plt.show()


## 4. Inference + GIF

In [ ]:
from src.inference import load_model, generate

model, word_to_id, id_to_word, device = load_model(C.CKPT_DIR / "best.pt")
print(f"Loaded model on {device}")
print(f"Vocab: {list(word_to_id.keys())}")


In [ ]:
for word in word_to_id:
    pose_flat = generate(model, word, word_to_id, device)
    pose_np = pose_tensor_to_numpy(pose_flat)
    out = render_animation(
        pose_np,
        C.OUT_DIR / f"gen_{word}.gif",
        title=f"Generated: {word}",
    )
    print(f"  {word:20s} -> {out.name}")

print("\nDone. outputs/ хавтсыг шалгана уу.")


## Дараагийн алхамууд

1. **Хэрэв сургалтын loss буурахгүй бол:** `data/pose_per_individual_videos/` доторх JSON-ийн бүтэц `dataset.py`-ийн хүлээж буйтай таарч буй эсэхийг шалгана.
2. **Хэрэв generated pose шуугиантай бол:** `SMOOTH_WEIGHT`-ийг `config.py`-д 0.5-1.0 болгож үзнэ.
3. **Илүү үг нэмэх:** `NUM_WORDS = 50` болгож, `EPOCHS`-ийг 200 хадгалж эхлэх.
4. **EchoMimicV2 руу шилжих:** generated `(50, 55, 2)` тэнсорыг DWPose форматад хөрвүүлж EchoMimicV2-н control input болгох.
